In [1]:
!pip install catboost
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"  # allows multiple prints from a cell

# install all packages and dependencies so that the code cell runs without errors.

import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os, plotly.express as px
# import xgboost as xgb, lightgbm, re, tensorflow as tf, tensorflow.keras as keras
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
import sklearn.preprocessing # trasnformers
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.neural_network import MLPRegressor  # SKLearn's MLP is optimised for CPU (and doesn't use GPU)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error as mse, r2_score
# other imports or implementation (for example, Robust Regressor with interpretable match-case statements)

np.set_printoptions(linewidth=10000, precision=2, edgeitems=20, suppress=True)
pd.set_option('display.max_colwidth', 100, 'display.max_columns', 10, 'display.width', 1000, 'display.max_rows', 8)

In [2]:
train_sample = pd.read_csv('train_sample.csv')
train_sample.head(3)

,start_point,end_point,time_of_day,day_of_week,traffic_condition,...,population_density,weather,public_transport_availability,historical_delay_factor,travel_time
0,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),day,Sunday,NaN,...,high,NaN,1,0.878909,26.907612
1,West Jakarta (Jakarta Barat),South Jakarta (Jakarta Selatan),morning,Thursday,NaN,...,high,NaN,1,1.081668,27.489129
2,Central Jakarta (Jakarta Pusat),East Jakarta (Jakarta Timur),morning,Thursday,NaN,...,low,NaN,2,1.192379,27.228978


In [4]:
test_sample = pd.read_csv('test_sample.csv')

In [5]:
fill_mode = lambda col: col.fillna(col.mode())
train_sample = train_sample.fillna({k: v[0] for k, v in train_sample.mode().to_dict().items()})
test_sample = test_sample.fillna({k: v[0] for k, v in train_sample.mode().to_dict().items()})

In [6]:
start = train_sample["start_point"].str.split().str[0]
end = train_sample["end_point"].str.split().str[0]

train_sample["start_end_point"] = start + " " + end
test_sample["start_end_point"] = test_sample["start_point"].str.split().str[0] + " " + test_sample["end_point"].str.split().str[0]

In [49]:
y_train = train_sample['travel_time']
X_train = train_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"]]
X_test = test_sample[[
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"]]
# etc.
# your code here

In [50]:
cat_cols = [
    "start_point",
    "end_point",
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [51]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0785634	total: 77.8ms	remaining: 23.3s
100:	learn: 4.7955460	total: 5.24s	remaining: 10.3s
200:	learn: 4.7269749	total: 9.83s	remaining: 4.84s
299:	learn: 4.7045380	total: 14.4s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0942582	total: 39.9ms	remaining: 11.9s
100:	learn: 4.7693384	total: 4.29s	remaining: 8.44s
200:	learn: 4.7199833	total: 8.77s	remaining: 4.32s
299:	learn: 4.6983183	total: 13.4s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9575673	total: 47.3ms	remaining: 14.2s
100:	learn: 4.6164142	total: 4.71s	remaining: 9.28s
200:	learn: 4.5291346	total: 9.01s	remaining: 4.44s
299:	learn: 4.4983824	total: 13.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0897340	total: 75.2ms	remaining: 22.5s
100:	learn: 4.7796355	total: 4.07s	remaining: 8.02s
200:	learn: 4.7080380	total: 8.57s	remaining: 4.22s
299:	learn: 4.6823608	total: 13s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0651410	total: 

In [15]:
y_train = train_sample['travel_time']
X_train = train_sample.drop(['travel_time', 'start_point', 'end_point'], axis=1)
X_test = test_sample
# etc.
# your code here

In [ ]:
cat_cols = [
    "time_of_day",
    "day_of_week",
    "vehicle_density",
    "population_density",
    "weather",
    "is_holiday",
    "start_end_point",
    "public_transport_availability"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [18]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0775478	total: 74.1ms	remaining: 22.1s
100:	learn: 4.6462028	total: 4.68s	remaining: 9.23s
200:	learn: 4.5715912	total: 9.1s	remaining: 4.48s
299:	learn: 4.5351920	total: 13.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0651501	total: 44ms	remaining: 13.1s
100:	learn: 4.6397320	total: 4.48s	remaining: 8.82s
200:	learn: 4.5661896	total: 8.79s	remaining: 4.33s
299:	learn: 4.5208564	total: 13s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9529493	total: 44.5ms	remaining: 13.3s
100:	learn: 4.4422226	total: 3.9s	remaining: 7.67s
200:	learn: 4.3778803	total: 8.19s	remaining: 4.03s
299:	learn: 4.3337025	total: 12.6s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0409397	total: 43.7ms	remaining: 13.1s
100:	learn: 4.6503674	total: 4.35s	remaining: 8.58s
200:	learn: 4.5614786	total: 8.38s	remaining: 4.13s
299:	learn: 4.5234089	total: 12.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0767578	total: 29.6

In [29]:
y_train = train_sample['travel_time']
X_train = train_sample[["start_end_point"]]
X_test = test_sample[["start_end_point"]]
# etc.
# your code here

In [30]:
cat_cols = [
    "start_end_point"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [31]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0772443	total: 33.5ms	remaining: 10s
100:	learn: 7.7045452	total: 3.89s	remaining: 7.67s
200:	learn: 7.7037206	total: 7.97s	remaining: 3.93s
299:	learn: 7.7034865	total: 11.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0918208	total: 35.3ms	remaining: 10.5s
100:	learn: 7.5731837	total: 3.7s	remaining: 7.28s
200:	learn: 7.5717203	total: 7.67s	remaining: 3.78s
299:	learn: 7.5711674	total: 11.6s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9560631	total: 21.5ms	remaining: 6.43s
100:	learn: 7.3588971	total: 3.09s	remaining: 6.08s
200:	learn: 7.3581110	total: 7.04s	remaining: 3.47s
299:	learn: 7.3578341	total: 11s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0883722	total: 40ms	remaining: 12s
100:	learn: 7.6319692	total: 3.83s	remaining: 7.54s
200:	learn: 7.6304465	total: 7.97s	remaining: 3.93s
299:	learn: 7.6300121	total: 12s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0621929	total: 39.4ms	re

In [40]:
y_train = train_sample['travel_time']
X_train = train_sample[["is_holiday"]]
X_test = test_sample[["is_holiday"]]
# etc.
# your code here

In [41]:
cat_cols = [
    "is_holiday",
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [42]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 15.0876090	total: 8.29ms	remaining: 2.48s
100:	learn: 15.0875767	total: 436ms	remaining: 859ms
200:	learn: 15.0875767	total: 741ms	remaining: 365ms
299:	learn: 15.0875767	total: 1.06s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.1100339	total: 10.4ms	remaining: 3.12s
100:	learn: 15.1099108	total: 320ms	remaining: 630ms
200:	learn: 15.1099108	total: 589ms	remaining: 290ms
299:	learn: 15.1099108	total: 855ms	remaining: 0us
Learning rate set to 0.188406
0:	learn: 14.9817532	total: 2.79ms	remaining: 833ms
100:	learn: 14.9817188	total: 282ms	remaining: 556ms
200:	learn: 14.9817188	total: 577ms	remaining: 284ms
299:	learn: 14.9817188	total: 874ms	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.1010267	total: 3.83ms	remaining: 1.15s
100:	learn: 15.1010264	total: 297ms	remaining: 585ms
200:	learn: 15.1010264	total: 587ms	remaining: 289ms
299:	learn: 15.1010264	total: 827ms	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.0

In [43]:
y_train = train_sample['travel_time']
X_train = train_sample[["population_density"]]
X_test = test_sample[["population_density"]]
# etc.
# your code here

In [44]:
cat_cols = [
    "population_density",
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [45]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 15.0860720	total: 34.8ms	remaining: 10.4s
100:	learn: 15.0757353	total: 3.86s	remaining: 7.61s
200:	learn: 15.0731129	total: 7.97s	remaining: 3.93s
299:	learn: 15.0720965	total: 12s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.1083651	total: 36.6ms	remaining: 11s
100:	learn: 15.1018832	total: 3.81s	remaining: 7.5s
200:	learn: 15.1005354	total: 7.82s	remaining: 3.85s
299:	learn: 15.0997709	total: 11.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 14.9803271	total: 37.4ms	remaining: 11.2s
100:	learn: 14.9749135	total: 3.75s	remaining: 7.4s
200:	learn: 14.9735399	total: 7.73s	remaining: 3.81s
299:	learn: 14.9726493	total: 11.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.0997714	total: 34.2ms	remaining: 10.2s
100:	learn: 15.0923702	total: 3.66s	remaining: 7.22s
200:	learn: 15.0905923	total: 7.7s	remaining: 3.79s
299:	learn: 15.0899142	total: 11.4s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.0902123	

In [46]:
y_train = train_sample['travel_time']
X_train = train_sample[["weather"]]
X_test = test_sample[["weather"]]
# etc.
# your code here

In [47]:
cat_cols = [
    "weather",
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [48]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 15.0875570	total: 28.2ms	remaining: 8.42s
100:	learn: 15.0836669	total: 3.41s	remaining: 6.73s
200:	learn: 15.0821617	total: 6.92s	remaining: 3.4s
299:	learn: 15.0811461	total: 10.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.1099540	total: 41.1ms	remaining: 12.3s
100:	learn: 15.1045100	total: 3.58s	remaining: 7.05s
200:	learn: 15.1030015	total: 7.19s	remaining: 3.54s
299:	learn: 15.1019747	total: 11s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 14.9816946	total: 31ms	remaining: 9.28s
100:	learn: 14.9792665	total: 3.59s	remaining: 7.08s
200:	learn: 14.9784213	total: 7.03s	remaining: 3.46s
299:	learn: 14.9778621	total: 11s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.1008199	total: 38.4ms	remaining: 11.5s
100:	learn: 15.0968098	total: 3.87s	remaining: 7.62s
200:	learn: 15.0952507	total: 7.79s	remaining: 3.84s
299:	learn: 15.0943658	total: 11.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 15.0920834	

In [ ]:
y_train = train_sample['travel_time']
X_train = train_sample[["start_end_point", "time_of_day"]]
X_test = test_sample[["start_end_point", "time_of_day"]]
# etc.
# your code here

In [ ]:
cat_cols = [
    "start_end_point",
    "time_of_day"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0786498	total: 32.5ms	remaining: 9.71s
100:	learn: 6.1885262	total: 3.86s	remaining: 7.6s
200:	learn: 6.1578778	total: 7.75s	remaining: 3.82s
299:	learn: 6.1503058	total: 11.7s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0936582	total: 35.6ms	remaining: 10.6s
100:	learn: 5.9947294	total: 3.99s	remaining: 7.87s
200:	learn: 5.9734981	total: 7.82s	remaining: 3.85s
299:	learn: 5.9594320	total: 11.6s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0800816	total: 39.9ms	remaining: 11.9s
100:	learn: 5.7354805	total: 3.6s	remaining: 7.09s
200:	learn: 5.6880527	total: 7.45s	remaining: 3.67s
299:	learn: 5.6640964	total: 11.3s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0912806	total: 39.5ms	remaining: 11.8s
100:	learn: 6.0502607	total: 4.17s	remaining: 8.22s
200:	learn: 6.0072607	total: 8.2s	remaining: 4.04s
299:	learn: 5.9944098	total: 12.3s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0651568	total: 3

In [ ]:
y_train = train_sample['travel_time']
X_train = train_sample[["start_end_point", "time_of_day", "day_of_week"]]
X_test = test_sample[["start_end_point", "time_of_day", "day_of_week"]]
# etc.
# your code here

In [ ]:
cat_cols = [
    "start_end_point",
    "time_of_day",
    "day_of_week"
]

model = CatBoostRegressor(
    iterations=300,
    depth=3,
    loss_function="RMSE",
    cat_features=tuple(cat_cols),
    verbose=100,
    random_seed=38
)

In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
    return_train_score=True
)

# print("CV RMSE:", -scores["test_score"].mean())

print('R²:', scores['test_r2'].mean())
print('MAE:', -scores['test_neg_mean_absolute_error'].mean())
print('MSE:', -scores['test_neg_mean_squared_error'].mean())

Learning rate set to 0.188406
0:	learn: 13.0768607	total: 45.4ms	remaining: 13.6s
100:	learn: 5.5686436	total: 4.12s	remaining: 8.12s
200:	learn: 5.3898568	total: 8.27s	remaining: 4.07s
299:	learn: 5.3350647	total: 12s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0904105	total: 23ms	remaining: 6.88s
100:	learn: 5.2645191	total: 3.93s	remaining: 7.74s
200:	learn: 5.1980633	total: 7.89s	remaining: 3.89s
299:	learn: 5.1549422	total: 11.9s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 12.9549079	total: 45.9ms	remaining: 13.7s
100:	learn: 5.0645541	total: 4.08s	remaining: 8.05s
200:	learn: 4.9580814	total: 8.04s	remaining: 3.96s
299:	learn: 4.9108161	total: 11.8s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0874977	total: 51.4ms	remaining: 15.4s
100:	learn: 5.3443003	total: 4.2s	remaining: 8.27s
200:	learn: 5.2002330	total: 8.24s	remaining: 4.06s
299:	learn: 5.1507713	total: 12.5s	remaining: 0us
Learning rate set to 0.188406
0:	learn: 13.0617195	total: 40.